# Part 1 - Data Cleaning with Python and pandas
**Project: CareLoop Clinics - No-Show Guard**

CareLoop runs 8 clinics. About 1 in 5 booked patients does not turn up (a "no-show").
In this notebook we take the raw appointment file, check it, clean it, and save a tidy
version that Part 2 (SQL), Part 3 (machine learning) and Part 4 (Gemini) will use.

| Step | What we do |
|---|---|
| 1.1 | Read the CSV with plain Python (`csv.DictReader`) and reject rows that fail a type check |
| 1.2 | Load with pandas, remove duplicates, clean text, parse dates |
| 1.3 | Fix invalid ages, fill missing values, cap outliers |
| 1.4 | Create new columns and do NumPy calculations |
| 1.5 | Call the Open-Meteo API for each clinic |
| 1.6 | Export the clean files and the SQL load script |

## Setup
The first cell installs the libraries (needed on Google Colab; on a laptop it finishes quickly
when they are already installed). The second cell finds the project folder so that the same
relative paths (`data/...`, `outputs/...`) work on a laptop and on Colab.

In [1]:
# Install the libraries this notebook needs (safe to re-run).
%pip install -q --disable-pip-version-check pandas numpy requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path

# On Google Colab the project files are not there yet, so we download (clone) the repo first.
if "google.colab" in str(get_ipython()) and not Path("MASAI-CAPSTONE-PROJ").exists():
    os.system("git clone -q https://github.com/Advay4706/MASAI-CAPSTONE-PROJ")
if Path("MASAI-CAPSTONE-PROJ").exists():
    os.chdir("MASAI-CAPSTONE-PROJ")

# This notebook lives in notebooks/. The data lives one level up, in data/.
# BASE is the project's top folder, wherever the notebook is started from.
BASE = Path(".") if Path("data").exists() else Path("..")
DATA = BASE / "data"
OUT = BASE / "outputs"
OUT.mkdir(exist_ok=True)   # create outputs/ if it does not exist

print("Data files found:", sorted(p.name for p in DATA.glob("*.csv")))

Data files found: ['appointments.csv', 'clinics.csv', 'followup_notes.csv']


## Step 1.1 - Read the raw file with `csv.DictReader` and reject bad rows

**What:** read `appointments.csv` using only Python's built-in `csv` module (no pandas yet).
Every value in a CSV file is text. `parse_row` tries to turn the number columns into real
numbers. If a conversion fails, Python raises a `ValueError`; we catch it with `try/except`
and put that row in a "rejected" list instead of crashing.

**Why:** this is how a real data pipeline protects itself - one bad row must not stop the
whole load, and rejected rows must be saved so somebody can look at them later.

**Important idea:** a *type check* only asks "is this a number?". It does **not** ask
"is this number sensible?". An age of `150` or `-4` is a perfectly good integer, so it
passes here. We deal with those in Step 1.3 (a *validity check*).

In [3]:
import csv
import json
from collections import Counter

# Which columns must be numbers, and which Python type each one should become.
NUMERIC_COLUMNS = {
    "age": int,              # whole number of years
    "distance_km": float,    # can have decimals, e.g. 12.5
    "fee_inr": float,        # consultation fee in rupees
    "prior_no_shows": int,   # count of earlier no-shows
}


def parse_row(row):
    """Convert one raw CSV row (all text) into a row with proper number types.

    Raises ValueError if a number column is blank or is not a number.
    The error message starts with the column name so we know what went wrong.
    """
    clean = dict(row)  # copy, so the original row is left untouched
    for column, to_type in NUMERIC_COLUMNS.items():
        text = row[column].strip()
        try:
            clean[column] = to_type(text)      # int("") or float("abc") raises ValueError
        except ValueError as err:
            # Raise it again with the column name added, so the caller can record the reason.
            raise ValueError(f"{column}: {err}") from err
    return clean


accepted = []   # rows that passed the type check
rejected = []   # rows that failed, with the reason
rows_read = 0

# newline="" is what the csv module documentation recommends when opening CSV files.
with open(DATA / "appointments.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)          # uses the header line as the dictionary keys
    # start=2 because line 1 of the file is the header
    for line_number, row in enumerate(reader, start=2):
        rows_read += 1
        try:
            accepted.append(parse_row(row))
        except ValueError as err:
            rejected.append({
                "line_number": line_number,
                "appointment_id": row["appointment_id"],
                "reason": str(err),
                "row": row,                 # keep the raw row exactly as it was
            })

# Save the rejected rows so they can be reviewed (never throw bad data away silently).
with open(OUT / "rejected_rows.json", "w", encoding="utf-8") as f:
    json.dump(rejected, f, indent=2)

print("Rows read    :", rows_read)
print("Rows accepted:", len(accepted))
print("Rows rejected:", len(rejected))

Rows read    : 1020
Rows accepted: 988
Rows rejected: 32


In [4]:
# Why were rows rejected? The reason text starts with the column name.
reasons = Counter(r["reason"].split(":")[0] for r in rejected)
print("Rejected by column:", dict(reasons))

print("\nExample rejected row:")
print(json.dumps(rejected[0], indent=2))

# A type check is not a validity check: impossible ages are still accepted here.
impossible = [r["age"] for r in accepted if r["age"] < 1 or r["age"] > 100]
print("\nImpossible ages that PASSED the type check:", impossible)
print("Types after parsing:", {c: type(accepted[0][c]).__name__ for c in NUMERIC_COLUMNS})

Rejected by column: {'distance_km': 20, 'age': 12}

Example rejected row:
{
  "line_number": 52,
  "appointment_id": "A0050",
  "reason": "distance_km: could not convert string to float: ''",
  "row": {
    "appointment_id": "A0050",
    "patient_id": "P117",
    "clinic_id": "C04",
    "booked_on": "2025-01-16",
    "appt_date": "2025-01-24",
    "age": "41",
    "gender": "M",
    "distance_km": "",
    "fee_inr": "600",
    "booking_channel": "Walk-in",
    "visit_type": "Consultation",
    "reminder_sent": "No",
    "prior_no_shows": "0",
    "status": "Attended"
  }
}

Impossible ages that PASSED the type check: [0, 150, -4, 250, 199]
Types after parsing: {'age': 'int', 'distance_km': 'float', 'fee_inr': 'float', 'prior_no_shows': 'int'}


**Reading the result**

- 1020 rows read, 988 accepted, 32 rejected. The 32 are 12 rows with a blank `age` and 20 rows
  with a blank `distance_km` (`int("")` and `float("")` both raise `ValueError`).
- The 5 impossible ages (0, 150, -4, 250, 199) were **accepted**, because each one is a valid
  integer. Catching them needs a business rule (age between 1 and 100), which comes in Step 1.3.
- Nothing is lost: the rejected rows sit in `outputs/rejected_rows.json`. In the pandas steps
  below we start again from the full raw file and *repair* these rows (fill the blanks)
  instead of dropping them.